<a href="https://colab.research.google.com/github/seo55555/dkdltmxl/blob/main/5%EC%A3%BC%EC%B0%A8_%EC%95%84%EC%9D%B4%EC%8A%A4%ED%8B%B0_%ED%8C%80%EA%B3%BC%EC%A0%9C.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

---
# 0. 환경 설정과 4주차 함수 재사용

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

pd.set_option('display.max_columns', 40)
pd.set_option('display.width', 170)
np.set_printoptions(suppress=True)

In [ ]:
import os
from google.colab import files
files.upload()

DATA, BOOK = 'knhanes_like_week4.csv', 'codebook_week4.csv'
df = pd.read_csv(DATA)
w  = df['wt'].to_numpy()
ST, PS = df['kstrata'].to_numpy(), df['psu'].to_numpy()

print(f'n = {len(df):,}  |  조사구 {df.psu.nunique()}개  |  층 {df.kstrata.nunique()}개')
print(f'가중치 합 = {w.sum():,.0f}')

Saving codebook_week4.csv to codebook_week4.csv
Saving knhanes_like_week4.csv to knhanes_like_week4.csv
n = 5,367  |  조사구 263개  |  층 18개
가중치 합 = 450,899


---
# 실습 ② — Table 1 완성

## 2-4. Table 1 조립  ★

**규칙** — n은 관측 수, %와 평균은 가중치 적용. 층화 열은 **도메인 분석**으로 만듭니다.

In [28]:
smk_exp = df.smk == 1  # 노출군 (현재흡연)
smk_unexp = df.smk == 0 if 'smk' in df and (df.smk == 0).any() else ~smk_exp  # 비노출군 (비흡연)

DOMS = {
    '전체': np.ones(len(df), dtype=bool),
    '노출군(흡연)': smk_exp,
    '비노출군(비흡연)': smk_unexp
}

# 2. 연속형 및 범주형 행 생성 함수 (n 열 제외)
def row_cont(label, var):
    """연속형: 가중 평균 (SE)"""
    r = {'특성': label}
    for name, dom in DOMS.items():
        th, se = svymean_domain(df[var], w, ST, PS, dom)  # svymean_domain 사용
        r[name] = f'{th:.1f} ({se:.2f})'
    return r

def row_cat(label, mask):
    """범주형: 가중 %"""
    mask = np.asarray(mask, float)
    r = {'특성': label}
    for name, dom in DOMS.items():
        th, se = svymean_domain(mask, w, ST, PS, dom)  # 지시변수의 도메인 가중평균
        r[name] = f'{th*100:.1f}'
    return r

# 3. Table 1 데이터프레임 생성
t1 = pd.DataFrame([
    row_cont('연령 (세)', 'age'),
    row_cat('  65세 이상', df.age >= 65),
    row_cat('남성', df.sex == 1),
    row_cat('읍면 거주', df.urban == 2),
    row_cont('BMI (kg/m²)', 'bmi'),
    row_cat('  비만 (BMI ≥ 25)', df.bmi >= 25),
    row_cont('수축기혈압 (mmHg)', 'sbp'),
    row_cat('고혈압', df.htn == 1),
    row_cat('현재흡연', df.smk == 1),
    row_cat('규칙적 운동', df.exercise == 2),
])
t1

,특성,전체,노출군(흡연),비노출군(비흡연)
0,연령 (세),49.1 (0.16),48.0 (0.39),49.5 (0.20)
1,65세 이상,16.8,14.4,17.6
2,남성,49.0,71.5,41.9
3,읍면 거주,22.9,23.6,22.7
4,BMI (kg/m²),23.8 (0.05),23.3 (0.11),24.0 (0.06)
5,비만 (BMI ≥ 25),36.5,30.7,38.4
6,수축기혈압 (mmHg),122.8 (0.45),124.0 (0.67),122.4 (0.46)
7,고혈압,24.4,27.2,23.5
8,현재흡연,24.1,100.0,0.0
9,규칙적 운동,41.5,42.1,41.3
